## Load data, create train test split based on years

In [ ]:
from scripts.load_clean_raw_data import load_stes_data, load_auxiliary_data
import pandas as pd

# Load stes training data in one df
data_dir = "data_raw/" 
stes_train = load_stes_data(data_dir, years=[2014, 2015, 2016, 2017, 2018])

# Load stes test data (year 2019)
stes_test = load_stes_data(data_dir, years=[2018, 2019])

# Load auxiliary data
aux_data = load_auxiliary_data(data_dir)

# load outcome (target) data
outcome_df = aux_data['outcome']

# add year column based on date column
outcome_df['monat'] = pd.to_datetime(outcome_df['monat'], format="%Y%m")
outcome_df['jahr'] = outcome_df['monat'].dt.year

# split outcomes in to train and test
outco_train = outcome_df[outcome_df["jahr"].isin([2014, 2015, 2016, 2017, 2018])]
outco_test = outcome_df[outcome_df["jahr"].isin([2018, 2019])]

## drop double rows
outco_train.drop_duplicates(inplace=True)
outco_test.drop_duplicates(inplace=True)

## Clean and preprocess data

In [ ]:
from scripts.load_clean_raw_data import clean_stes_data, clean_asal_data
from scripts.feature_engineering_job import engineer_job_features

# clean stes data
stes_train = clean_stes_data(stes_train)
stes_test = clean_stes_data(stes_test)

# load asal data
asal = aux_data['asal']

# clean asal data
asal = clean_asal_data(asal)

# load beruf data
beruf = aux_data['beruf']

# load avam sbn 2000 data
avam_sbn2000 = aux_data['avam_sbn2000']

# feature engineering for job data
beruf = engineer_job_features(beruf, avam_sbn2000)

# load arbfo data
arbfo = aux_data['arbeitsformen']

# clean arbfo data
arbfo.drop_duplicates(subset='pseudoSTESID', inplace=True)

# load regio data
regio = aux_data['regio']

# preprocess regio data (reduction of values to first two letters, which yields grossregionen & kantone)
regio["code_suchreg"] = regio["code_suchreg"].str[:2]
regio.drop_duplicates(subset='pseudoSTESID', inplace=True)

## Merge data

In [ ]:
from scripts.merge_raw_data import validate_merge_size, merge_stes_asal

stes_outco_train = pd.merge(outco_train, stes_train, on=['monat', 'pseudoPID'], how='left')
stes_outco_test =  pd.merge(outco_test, stes_test, on=['monat', 'pseudoPID'], how='left')

validate_merge_size(stes_outco_train, outco_train, stes_train)
validate_merge_size(stes_outco_test, outco_test, stes_test)

stes_outco_asal_train = merge_stes_asal(stes_outco_train, asal)
stes_outco_asal_test = merge_stes_asal(stes_outco_test, asal)

validate_merge_size(stes_outco_asal_train, stes_outco_train, asal)
validate_merge_size(stes_outco_asal_test, stes_outco_test, asal)

stes_outco_asal_beruf_train = pd.merge(stes_outco_asal_train, beruf, on=['pseudoSTESID', 'pseudoPID'], how='inner')
stes_outco_asal_beruf_test = pd.merge(stes_outco_asal_test, beruf, on=['pseudoSTESID', 'pseudoPID'], how='inner')

validate_merge_size(stes_outco_asal_beruf_train, stes_outco_asal_train, beruf)
validate_merge_size(stes_outco_asal_beruf_test, stes_outco_asal_test, beruf)

stes_outco_asal_beruf_arbfo_train = pd.merge(stes_outco_asal_beruf_train, arbfo, on=['pseudoSTESID', 'pseudoPID'], how='left')
stes_outco_asal_beruf_arbfo_test = pd.merge(stes_outco_asal_beruf_test, arbfo, on=['pseudoSTESID', 'pseudoPID'], how='left')

validate_merge_size(stes_outco_asal_beruf_arbfo_train, stes_outco_asal_beruf_train, arbfo)
validate_merge_size(stes_outco_asal_beruf_arbfo_test, stes_outco_asal_beruf_test, arbfo)

stes_outco_asal_beruf_arbfo_regio_train = pd.merge(stes_outco_asal_beruf_arbfo_train, regio, on=['pseudoSTESID', 'pseudoPID'], how='inner')
stes_outco_asal_beruf_arbfo_regio_test = pd.merge(stes_outco_asal_beruf_arbfo_test, regio, on=['pseudoSTESID', 'pseudoPID'], how='inner')

validate_merge_size(stes_outco_asal_beruf_arbfo_regio_train, stes_outco_asal_beruf_train, regio)
validate_merge_size(stes_outco_asal_beruf_arbfo_regio_test, stes_outco_asal_beruf_test, regio)

## Feature engineering and binning

In [ ]:
from scripts.feature_engineering import feature_engineering_dummies, feature_engineering_no_dummies

train_raw = stes_outco_asal_beruf_arbfo_regio_train
test_raw = stes_outco_asal_beruf_arbfo_regio_test

train_clean_dummies = feature_engineering_dummies(train_raw)
test_clean_dummies = feature_engineering_dummies(test_raw)

train_clean_no_dummies = feature_engineering_no_dummies(train_raw)
test_clean_no_dummies = feature_engineering_no_dummies(test_raw)

## Save clean train and test data (with and without dummies)

In [ ]:
train_clean_dummies.to_csv('train_test_clean/train_clean_dummies.csv')
test_clean_dummies.to_csv('train_test_clean/test_clean_dummies.csv')

train_clean_no_dummies.to_csv('train_test_clean/train_clean_no_dummies.csv')
test_clean_no_dummies.to_csv('train_test_clean/test_clean_no_dummies.csv')